# 10. Mediation & Sensitivity Analysis

## Background

Estimating a total treatment effect is often not enough. We want to know *how* the effect works: how much of the effect of education on wages operates through occupation choice (the mediator) versus other pathways? **Mediation analysis** decomposes the total effect into a natural direct effect (NDE) and a natural indirect effect (NIE).

Even when satisfied with our total effect estimate, a careful analyst asks: "how much unmeasured confounding would it take to explain away this result?" **Sensitivity analysis** answers this systematically. Rosenbaum's Gamma and E-values (VanderWeele & Ding 2017) are now required in rigorous observational studies.

## What You'll Learn

- The mediation decomposition: TE = NDE + NIE under sequential ignorability
- Product-of-coefficients method (linear case)
- Rosenbaum sensitivity analysis: what Gamma would reverse the conclusion?
- E-values: minimum confounding association to explain away the observed effect
- Sensitivity plots and how to communicate fragility to stakeholders

## Why This Matters

Mediation is common in biology (mechanism studies), marketing (attribution), and policy evaluation. The sensitivity framework is increasingly a publication requirement for observational studies. The E-value has become a practical standard because it is easily computed from a treatment effect estimate and its confidence interval — no simulation needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Mediation: Direct and Indirect Effects

**Total Effect** = NDE + NIE

- **Natural Direct Effect (NDE)**: effect of T on Y *not* through M
- **Natural Indirect Effect (NIE)**: effect of T on Y *through* M

Identification requires sequential ignorability:
1. (Y(t,m), M(t)) perp T | X — no T-Y or T-M confounding
2. Y(t,m) perp M | (T, X) — no M-Y confounding after conditioning on T

In [ ]:
def simulate_mediation(n=2000, nde=2.0, alpha_tm=0.6, seed=0):
    rng = np.random.default_rng(seed)
    X1 = rng.normal(0, 1, n)
    X2 = rng.binomial(1, 0.4, n)
    T = rng.binomial(1, 0.5, n)
    M = alpha_tm * T + 0.5*X1 + rng.normal(0, 1, n)
    beta_my = 1.0
    Y = nde * T + beta_my * M + 0.3*X1 + 0.5*X2 + rng.normal(0, 1, n)
    true_nie = alpha_tm * beta_my
    return pd.DataFrame({'T':T,'M':M,'Y':Y,'X1':X1,'X2':X2}), nde, true_nie


df_med, true_nde, true_nie = simulate_mediation(n=2000)
print(f"True NDE = {true_nde:.3f}")
print(f"True NIE = {true_nie:.3f}  (alpha_TM x beta_MY = 0.6 x 1.0)")
print(f"True TE  = {true_nde + true_nie:.3f}")

## 2. Product-of-Coefficients Method (Linear Case)

For linear models the mediation estimators are:
- **alpha_TM**: coefficient of T in  M ~ T + X
- **beta_MY**: coefficient of M in   Y ~ T + M + X
- **NIE** = alpha_TM x beta_MY
- **NDE** = coefficient of T in      Y ~ T + M + X

In [ ]:
def linear_mediation(df, mediator='M', outcome='Y', treatment='T', controls=None):
    ctrl_str = '' if not controls else ' + ' + ' + '.join(controls)
    m1 = smf.ols(f'{mediator} ~ {treatment}{ctrl_str}', data=df).fit()
    alpha_tm = m1.params[treatment]; se_alpha = m1.bse[treatment]
    m2 = smf.ols(f'{outcome} ~ {treatment} + {mediator}{ctrl_str}', data=df).fit()
    beta_my = m2.params[mediator]; nde_hat = m2.params[treatment]
    se_beta = m2.bse[mediator];   se_nde  = m2.bse[treatment]
    nie_hat = alpha_tm * beta_my
    te_hat  = nde_hat + nie_hat
    se_nie  = np.sqrt(beta_my**2 * se_alpha**2 + alpha_tm**2 * se_beta**2)
    print(f"{'Effect':<25} {'Estimate':>10} {'SE':>8} {'95% CI':>20}")
    print("-" * 65)
    for label, est, se in [
        ('NDE (direct)',  nde_hat, se_nde),
        ('NIE (indirect)',nie_hat, se_nie),
        ('TE (total)',    te_hat,  np.sqrt(se_nde**2 + se_nie**2)),
    ]:
        lo, hi = est - 1.96*se, est + 1.96*se
        print(f"{label:<25} {est:>10.3f} {se:>8.3f} [{lo:>6.3f}, {hi:>6.3f}]")
    return dict(nie=nie_hat, nde=nde_hat, te=te_hat,
                alpha_tm=alpha_tm, beta_my=beta_my,
                pct_mediated=nie_hat/te_hat)


result = linear_mediation(df_med, controls=['X1','X2'])
print(f"\nPercent mediated: {result['pct_mediated']:.1%}")

In [ ]:
# Path diagram
fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')
nodes = {'T': (0.1, 0.5), 'M': (0.5, 0.8), 'Y': (0.9, 0.5)}
for name, (x, y) in nodes.items():
    ax.add_patch(plt.Circle((x, y), 0.07, color='#E3F2FD', ec='#1565C0', lw=2))
    ax.text(x, y, name, ha='center', va='center', fontsize=14, fontweight='bold')

def arrow(ax, frm, to, label, color='black', offset=(0,0)):
    x0, y0 = nodes[frm]; x1, y1 = nodes[to]
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5,
                               connectionstyle='arc3,rad=0.2'))
    ax.text((x0+x1)/2+offset[0], (y0+y1)/2+offset[1], label,
            ha='center', va='center', fontsize=11, color=color,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=color, alpha=0.8))

arrow(ax, 'T', 'M', f"a={result['alpha_tm']:.2f}", '#F44336', (0, 0.08))
arrow(ax, 'M', 'Y', f"b={result['beta_my']:.2f}", '#F44336', (0, 0.08))
arrow(ax, 'T', 'Y', f"NDE={result['nde']:.2f}", '#2196F3', (0, -0.15))
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title(f"Mediation Path Diagram — NIE = a*b = {result['nie']:.3f} ({result['pct_mediated']:.0%} mediated)")
plt.tight_layout()
plt.savefig('10_mediation_diagram.png', dpi=110, bbox_inches='tight')
plt.show()

## 3. Rosenbaum Sensitivity Analysis

Gamma (Γ) is the odds ratio for treatment selection due to unmeasured confounding. Γ=1 = no hidden bias. We find the critical Γ at which the result is no longer statistically significant.

In [ ]:
def rosenbaum_sensitivity(y_treated, y_control, gamma_range=None, alpha=0.05):
    if gamma_range is None:
        gamma_range = np.arange(1.0, 3.05, 0.1)
    rng = np.random.default_rng(42)
    n_min = min(len(y_treated), len(y_control))
    idx_t = rng.choice(len(y_treated), n_min, replace=False)
    idx_c = rng.choice(len(y_control), n_min, replace=False)
    diffs = y_treated[idx_t] - y_control[idx_c]
    n = len(diffs)
    ranks = np.arange(1, n+1)
    sorted_idx = np.argsort(np.abs(diffs))
    ranks_sorted = np.zeros(n)
    ranks_sorted[sorted_idx] = ranks
    W_obs = np.sum(ranks_sorted[diffs > 0])
    results = []
    for gamma in gamma_range:
        p_plus = gamma / (1 + gamma)
        p_minus = 1 / (1 + gamma)
        E_max = p_plus * n*(n+1)/2
        V_max = p_plus * p_minus * n*(n+1)*(2*n+1)/6
        z_max = (W_obs - E_max) / np.sqrt(V_max)
        p_max = 1 - stats.norm.cdf(z_max)
        results.append({'gamma': gamma, 'p_upper': p_max})
    return pd.DataFrame(results)


np.random.seed(1)
y_t = np.random.normal(7, 2, 200)
y_c = np.random.normal(5, 2, 200)
sens = rosenbaum_sensitivity(y_t, y_c)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sens.gamma, sens.p_upper, 'o-', color='#2196F3', lw=2)
ax.axhline(0.05, color='red', ls='--', lw=1.5, label='alpha=0.05')
ax.fill_between(sens.gamma, 0, sens.p_upper,
                where=sens.p_upper < 0.05, alpha=0.2, color='green',
                label='Significant at this Gamma')
critical_gamma = sens.loc[sens.p_upper <= 0.05, 'gamma'].max()
ax.axvline(critical_gamma, color='gray', ls=':', lw=1.5,
           label=f'Critical Gamma = {critical_gamma:.1f}')
ax.set_xlabel('Gamma (unmeasured confounding odds ratio)')
ax.set_ylabel('Worst-case p-value')
ax.set_title(f'Rosenbaum Sensitivity Analysis — robust up to Gamma={critical_gamma:.1f}')
ax.legend()
plt.tight_layout()
plt.savefig('10_rosenbaum.png', dpi=110, bbox_inches='tight')
plt.show()
print(f"Critical Gamma = {critical_gamma:.1f}")
print("An unmeasured confounder would need to multiply treatment odds by")
print(f"{critical_gamma:.1f}x to make the result non-significant.")

## 4. E-values

The E-value (VanderWeele & Ding 2017) is the minimum strength of unmeasured confounding on the risk-ratio scale needed to explain away the result. No simulation required: E = RR + sqrt(RR * (RR - 1)).

In [ ]:
def evalue(rr, rr_lo=None):
    def ev(r):
        if r < 1: r = 1/r
        if r == 1: return 1.0
        return r + np.sqrt(r * (r - 1))
    return {'rr': rr, 'evalue_point': ev(rr),
            'evalue_ci': ev(rr_lo) if rr_lo else None}


examples = [
    ('Weak association (RR=1.2)', 1.2, 1.05),
    ('Moderate (RR=2.0)',         2.0, 1.5),
    ('Strong (RR=4.0)',           4.0, 2.8),
    ('Very strong (RR=8.0)',      8.0, 5.0),
]

print(f"{'Study':<35} {'RR':>5} {'E-value':>10} {'E-value (CI)':>14}")
print("-" * 70)
for label, rr, rr_lo in examples:
    ev = evalue(rr, rr_lo)
    print(f"{label:<35} {rr:>5.1f} {ev['evalue_point']:>10.2f} {ev['evalue_ci']:>14.2f}")

print("\nE-value interpretation: an unmeasured confounder must be associated")
print("with BOTH treatment AND outcome by at least E-value fold to explain away")
print("the result. E=2 is fragile; E=5 is robust.")

In [ ]:
rr_vals = np.linspace(1.0, 8.0, 100)
ev_vals = [evalue(r)['evalue_point'] for r in rr_vals]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rr_vals, ev_vals, color='#2196F3', lw=2.5)
for label, rr, _ in examples:
    ev = evalue(rr)['evalue_point']
    ax.scatter([rr], [ev], s=80, color='#F44336', zorder=5)
    ax.text(rr+0.1, ev+0.1, f'RR={rr}
E={ev:.1f}', fontsize=8)
ax.set_xlabel('Observed Risk Ratio'); ax.set_ylabel('E-value')
ax.set_title('E-value as a Function of Observed RR')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('10_evalue.png', dpi=110, bbox_inches='tight')
plt.show()